# **Assignment 13**

In [1]:
import os
import numpy as np
import torch
import torch.nn as nn
import pandas as pd
import cv2
import mediapipe as mp
import pandas as pd
import numpy as np
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
from IPython.display import HTML
import joblib
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import f1_score
import torch.optim as optim
from sklearn.model_selection import KFold
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    roc_auc_score
)

if torch.backends.mps.is_available():
    device = torch.device("mps")      # Mac GPU (Apple Silicon)
elif torch.cuda.is_available():
    device = torch.device("cuda")     # Nvidia GPU
else:
    device = torch.device("cpu")

## Load cut and padded squat sequences with target determining if squat good or bad

In [2]:
df = pd.read_csv("../../MainProject/data/mediapipe_padded_videos/G01_padded.csv")

y = df["target"].values

X_flat = df.drop(columns=["target"]).values

max_frames = 173 
n_features = 39

X = X_flat.reshape(-1, max_frames, n_features)

print(X.shape)

print(X)
print(y)

(1, 173, 39)
[[[ 0.52720726  0.16865908 -0.32241583 ...  0.49026176  0.78741807
    0.16503   ]
  [ 0.52720386  0.16914926 -0.33219615 ...  0.48986819  0.78740412
    0.15604694]
  [ 0.52691746  0.1691364  -0.32683185 ...  0.4899978   0.792409
    0.15246266]
  ...
  [ 0.          0.          0.         ...  0.          0.
    0.        ]
  [ 0.          0.          0.         ...  0.          0.
    0.        ]
  [ 0.          0.          0.         ...  0.          0.
    0.        ]]]
[1]


## Define functions

In [ ]:
# Define dense squat classifying model
class SquatClassifierDense(nn.Module):
    def __inint__(self, input_dim, hidden_layers: list, activation="relu", dropout=0.0):
        super().__init__()

        layers = []
        activations = {"relu": nn.ReLU(),
                       "tanh": nn.Tanh(),
                       "gelu": nn.GELU(),
                       "leaky_relu": nn.LeakyReLU()
                       }
        
        prev_size = input_size

        for hidden_size in hidden_layers:
            layers.append(nn.Linear(prev_size, hidden_size))
            layers.append(activations[activation]())

            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            
            prev_size = hidden_size

        # Output layer
        layers.append(nn.Linear(prev_size, 1))

        self.network = nn.Sequential(*layers)
        self.network.apply(init_weights)

    def forward(self, x):
        return self.network(x)


def build_dense_model(config, input_size):
    return SquatClassifierDense(
        input_size=input_size,
        hidden_layers=config["layers"],
        activation=config["activation"],
        dropout=config["dropout"]
    ).to(device)


# Define initial weights and biases
def init_weights(m):
    if isinstance(m, nn.Linear):
        nn.init.kaiming_uniform_(m.weight) # good for ReLU
        nn.init.zeros_(m.bias)


# Compute metrics
def compute_metrics(y_true, probs, threshold=0.5):

    preds = (probs >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_true, preds).ravel()

    accuracy = accuracy_score(y_true, preds)
    precision = precision_score(y_true, preds, zero_division=0)
    recall = recall_score(y_true, preds, zero_division=0)

    # AUC
    auc = roc_auc_score(y_true, probs)

    return {
        "tp": tp,
        "fp": fp,
        "tn": tn,
        "fn": fn,
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "auc": auc
    }

# Full batch since it's faster on my cpu
def train_one_model(model, config, x_train, y_train, x_val, y_val, loss_fn):

    optimizer = optim.Adam(model.parameters(), lr=config["lr"])
    epochs = config["epochs"]

    best_val_auc = 0
    best_state = None

    patience = 10
    epochs_no_improve = 0

    # Convergence tracking
    history = {
        "train_loss": [],
        "val_loss": [],
        "val_auc": []
    }

    for epoch in range(epochs):

        # Training
        model.train()
        optimizer.zero_grad()

        logits = model(x_train)
        train_loss = loss_fn(logits, y_train)

        train_loss.backward()
        optimizer.step()

        # Validation with auc as main metric
        model.eval()
        with torch.no_grad():
            val_logits = model(x_val)

            probs = torch.sigmoid(val_logits)

            val_auc = roc_auc_score(
            y_val.cpu().numpy(),
            probs.cpu().numpy()
            )

        # Store history for each epoch
        history["train_loss"].append(train_loss.item())
        history["val_loss"].append(val_loss.item())
        history["val_auc"].append(val_auc)

        # Early stopping
        if val_auc > best_val_auc:
            best_val_auc = val_auc
            best_state = model.state_dict()
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1

        if epochs_no_improve >= patience:
            break

    # Load best weights
    model.load_state_dict(best_state)

    return best_val_auc, model, history


# Cross validation
def cross_validate_model(config, X, y, loss_fn, n_splits=10):

    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

    fold_auc_scores = []
    fold_histories = []

    for fold, (train_idx, val_idx) in enumerate(kf.split(X)):
        # Split data
        X_train, X_val = X[train_idx], X[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]

        # Convert to torch tensors
        X_train = torch.tensor(X_train, dtype=torch.float32)
        X_val   = torch.tensor(X_val, dtype=torch.float32)

        y_train = torch.tensor(y_train, dtype=torch.float32)
        y_val   = torch.tensor(y_val, dtype=torch.float32)

        # Build fresh model
        model = build_model(config)

        # Train
        val_auc, model, history = train_one_model(
            model, config,
            X_train, y_train,
            X_val, y_val,
            loss_fn
        )

        fold_auc_scores.append(val_auc)
        fold_histories.append(history)

        print(f"Fold {fold+1} F1: {val_f1:.4f}")

    # Aggregate results
    mean_auc = np.mean(fold_auc_scores)
    std_auc  = np.std(fold_auc_scores)

    print(f"Mean AUC: {mean_auc:.4f} ± {std_auc:.4f}")

    return mean_f1, std_f1, fold_f1_scores, fold_histories